# Food Energy Water Nexus  (FEWN)

__author__ = "Rahul Kakodkar"
__copyright__ = "Copyright 2023, Multi-parametric Optimization & Control Lab"
__credits__ = ["Rahul Kakodkar", "Marcello Di Martino", "Efstratios N. Pistikopoulos"]
__license__ = "Open"
__version__ = "1.0.0"
__maintainer__ = "Rahul Kakodkar"
__email__ = "cacodcar@tamu.edu"
__status__ = "Production"


$\textbf{Import modules}$

In [1]:
import pandas 
from numpy import poly1d, polyfit, arange
from src.energiapy.components.temporal_scale import Temporal_scale
from src.energiapy.components.resource import Resource
from src.energiapy.components.process import Process, Costdynamics
from src.energiapy.components.material import Material
from src.energiapy.components.location import Location
from src.energiapy.components.network import Network
from src.energiapy.components.scenario import Scenario
from src.energiapy.components.transport import Transport
from src.energiapy.components.result import Result 
from src.energiapy.model.formulate_milp import formulate_milp
from src.energiapy.utils.data_utils import get_data, make_henry_price_df
from src.energiapy.utils.nsrdb_utils import fetch_nsrdb_data
from src.energiapy.plot import plot
from src.energiapy.model.solve import solve
from src.energiapy.utils.data_utils import load_results
import matplotlib.pyplot as plt
from matplotlib import rc
from itertools import product

**Import solar dni and wind speeds for Harris county**


In [2]:
weather20_df = pandas.read_csv('data/ho_solar20.csv', index_col=0)
weather20_df.index = [i.split('+')[0] for i in weather20_df.index]
weather = weather20_df[~weather20_df.index.str.contains('02-29')] #remove leap years
weather

,wind_speed,dni
2020-01-01 00:00:00,9.5,0.0
2020-01-01 01:00:00,7.5,0.0
2020-01-01 02:00:00,6.0,0.0
2020-01-01 03:00:00,6.0,0.0
2020-01-01 04:00:00,6.0,0.0
...,...,...
2020-12-31 19:00:00,54.5,0.5
2020-12-31 20:00:00,55.5,0.0
2020-12-31 21:00:00,50.0,0.0
2020-12-31 22:00:00,46.0,181.5


**import cost data**


$\textbf{Define temporal scale}$


In [3]:
scales = Temporal_scale(discretization_list=[1, 365, 24], start_zero= 2020)

$\textbf{Declare constants for ease}$


In [4]:
bigM = 10**4  # very large number
smallM = 0.1
water_price = 31.70  # $/5000gallons
power_price = 8  # cents/kWh
ur_price = 42.70  # 250 Pfund U308 (Uranium)
A_f = 0.05  # annualization factor
# CO2_res = 0.2
pv_start = 0
ake_start = 0
smrh_start = 0
smr_start = 0
asmr_start = 0

$\textbf{Declare resources}$

In [5]:
Charge = Resource(name='Charge', sell=False,
                  store_max=100, basis='MW', label='Battery energy', block='energystorage')

Solar = Resource(
    name='Solar', cons_max=bigM, basis='MW', label='Solar Power', block='energyfeedstock')

Wind = Resource(name='Wind', cons_max= bigM, basis='MW', label='Wind Power', block='energyfeedstock')


H2O = Resource(name='H2O', cons_max=bigM, price= water_price/(5000*3.7854), basis='kg', label='Water', block='Resource')

Biomass = Resource(name = 'Biomass', basis = 'MW', label = 'Biomass consumed')

Maize = Resource(name='Maize', basis='acre',
                 label='Maize cultivated', block='Resource', cons_max = bigM)

Power = Resource(name='Power', basis='MW',
                 label='Renewable power generated', demand = True, sell = True, block='Resource')


$\textbf{Declare processes}$

In [6]:
Battery1_c = Process(name='Battery1_c', conversion={Charge: 1, Power: -1}, cost = {'CAPEX_capacity': 0.152, 'CAPEX_power': 0.292, 'units': '$/MWh'},\
    prod_max=bigM, block='power_storage', lifetime = 50, label='Battery type 1', costdynamics= Costdynamics.battery)

Battery1_d = Process(name='Battery1_d', conversion={Charge: -1, Power: 1}, cost = {'CAPEX_capacity': 0.001, 'CAPEX_power': 0.001, 'units': '$/MWh'},\
    prod_max=bigM, block='power_storage',  lifetime = 50, label='Battery type 1', costdynamics= Costdynamics.battery)


Battery2_c = Process(name='Battery2_c', conversion={Charge: 1, Power: -1}, cost = {'CAPEX_capacity': 0.152, 'CAPEX_power': 0.292, 'units': '$/MWh'},\
    prod_max=bigM, block='power_storage',  lifetime = 30, label='Battery type 2', costdynamics= Costdynamics.battery)

Battery2_d = Process(name='Battery2_d', conversion={Charge: -1, Power: 1}, cost = {'CAPEX_capacity': 0.001, 'CAPEX_power': 0.001, 'units': '$/MWh'},\
    prod_max=bigM, block='power_storage',  lifetime = 30, label='Battery type 2', costdynamics= Costdynamics.battery)

WF = Process(name='WF', conversion={Wind: -1, Power: 1}, cost= {'CAPEX': 2009.5/20, 'Incidental': 0 },
             prod_max=bigM, land= 10800/1800, block='power_generation', costdynamics= Costdynamics.wind,
             label='Wind mill array', citation='Use windtoolkit conversion')

# PV = Process(name='PV', conversion={Solar: -1, Power: 1}, cost= {'CAPEX': 4726.7/20, 'Incidental': 519.34/20},
#              prod_max=bigM, land= 13320/1800, block='power_generation', costdynamics= Costdynamics.solar,\
#                  label='Solar photovoltaics (PV) array', citation='Fixed angle tracking')


PV = Process(name='PV', conversion={Solar: -1, Power: 1}, cost= {'CAPEX': 4726.7/20, 'Incidental': 519.34/20},
             prod_max=bigM, land= 10000/1800, block='power_generation', costdynamics= Costdynamics.solar,\
                 label='Solar photovoltaics (PV) array', citation='Fixed angle tracking')

Farm = Process(name= 'Farm', conversion={Maize: -1, Power:1}, cost =  {'CAPEX': 4440/15, 'Fixed O&M': 4.221*10**(-3),\
    'Variable O&M': 0, 'units': '$/acre','source': 'dummy'}, label = 'Agriculture')

# Bioreactor = Process(name = 'Bioreactor', conversion= {Biomass:-1, Power: 1}, cost ={'CAPEX': 1, 'Fixed O&M': 10**(-3),\
#     'Variable O&M': 0, 'units': '$/acre','source': 'dummy'} , label = 'Yeehaw')

In [7]:
process_set = {Battery1_c, Battery1_d, Battery2_c, Battery2_d, WF, PV, Farm}

In [8]:
WF.capex

100.475

$\textbf{Declare location(s)}$


In [9]:
Place = Location(name='Place', processes= process_set, \
        capacity_factor = {PV: pandas.DataFrame(weather['dni']), WF: pandas.DataFrame(weather['wind_speed'])}, scales=scales, \
        label='Place', demand_level=2, capacity_level= 2, cost_level= 1)

In [10]:
# plot.capacity_factor(location= HO, process= PV, color= 'orange')
# plot.capacity_factor(location= HO, process= WF, color= 'blue')
# plot.cost_factor (location= HO, resource= CH4, color= 'red')
# plot.demand_factor (location= HO, resource= Mile)

In [11]:
scenario = Scenario(name= 'shell', network= Place, scales= scales,  expenditure_scale_level= 2, scheduling_scale_level= 2, \
    network_scale_level= 0, demand_scale_level= 2, label= 'shell milp case study (HO)')

In [12]:
scenario.demand_factor

{'Place': None}

$\textbf{Formulate model}$

A pyomo instance is formulated from the scenario

Concises sets and corresponding variables are declared.

Corresponding constraints are generated based on the nature of model chosen

In the presented example, a MILP is formulated



In [13]:
from pyomo.environ import ConcreteModel
from src.energiapy.model.sets import generate_sets
from src.energiapy.model.variables import *
from src.energiapy.model.constraints import *
from src.energiapy.model.objectives import cost_objective, uncertainty_cost_objective

def formulate_houston_milp(scenario: Scenario, carbon_bound:float= None, carbon_reduction_percentage:float= 0, \
    demand: float = 1000, land_restriction: float = 0) -> ConcreteModel:
    """formulates a multi-scale mixed integer linear programming formulation of the scenario

    Args:
        scenario (Scenario): scenario under consideration

    Returns:
        ConcreteModel: pyomo model instance with sets, variables, constraints, objectives generated
    """

    instance = ConcreteModel()

    generate_sets(instance=instance, location_set=scenario.location_set, transport_set=scenario.transport_set, scales=scenario.scales,
                  process_set=scenario.process_set, resource_set=scenario.resource_set, material_set=scenario.material_set,
                  source_set=scenario.source_locations, sink_set=scenario.sink_locations)

    generate_scheduling_vars(
        instance=instance, scale_level=scenario.scheduling_scale_level)
    generate_network_vars(
        instance=instance, scale_level=scenario.network_scale_level)
    generate_network_binary_vars(
        instance=instance, scale_level=scenario.network_scale_level)

    if len(instance.locations) > 1:
        generate_transport_vars(
            instance=instance, scale_level=scenario.scheduling_scale_level)

    demand_constraint(instance=instance, demand_scale_level=scenario.demand_scale_level,
                      scheduling_scale_level=scenario.scheduling_scale_level, demand = demand, demand_factor=scenario.demand_factor)
    
    inventory_balance_constraint(instance=instance, scheduling_scale_level=scenario.scheduling_scale_level,
                                 conversion=scenario.conversion)
    nameplate_production_constraint(instance=instance, capacity_factor=scenario.capacity_factor,
                                    network_scale_level=scenario.network_scale_level, scheduling_scale_level=scenario.scheduling_scale_level)
    nameplate_inventory_constraint(instance=instance, loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level,
                                   scheduling_scale_level=scenario.scheduling_scale_level)
    resource_consumption_constraint(instance=instance, loc_res_dict=scenario.loc_res_dict,
                                    cons_max=scenario.cons_max, scheduling_scale_level=scenario.scheduling_scale_level)
    resource_purchase_constraint(instance=instance, cost_factor=scenario.cost_factor, price=scenario.price,
                                 loc_res_dict=scenario.loc_res_dict, scheduling_scale_level=scenario.scheduling_scale_level,
                                 expenditure_scale_level=scenario.expenditure_scale_level)
    # resource_discharge_constraint(instance= instance, scheduling_scale_level= scenario.scheduling_scale_level)

    production_facility_constraint(instance=instance, prod_max=scenario.prod_max,
                                   loc_pro_dict=scenario.loc_pro_dict, network_scale_level=scenario.network_scale_level)
    storage_facility_constraint(instance=instance, store_max=scenario.store_max,
                                loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level)

    min_production_facility_constraint(instance=instance, prod_min=scenario.prod_min,
                                       loc_pro_dict=scenario.loc_pro_dict, network_scale_level=scenario.network_scale_level)
    min_storage_facility_constraint(instance=instance, store_min=scenario.store_min,
                                    loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level)

    location_production_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_discharge_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_consumption_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_purchase_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)

    network_production_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_discharge_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_consumption_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_purchase_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)

    process_capex_constraint(instance=instance, capex_dict=scenario.capex_dict,
                             network_scale_level=scenario.network_scale_level)
    process_fopex_constraint(instance=instance, fopex_dict=scenario.fopex_dict,
                             network_scale_level=scenario.network_scale_level)
    process_vopex_constraint(instance=instance, vopex_dict=scenario.vopex_dict,
                             network_scale_level=scenario.network_scale_level)

    process_incidental_constraint(instance=instance, incidental_dict=scenario.incidental_dict,
                             network_scale_level=scenario.network_scale_level)

    process_land_constraint(instance=instance, land_dict=scenario.land_dict,
                            network_scale_level=scenario.network_scale_level)
    location_land_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_land_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)

    location_land_restriction_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, land_restriction= land_restriction)

    location_capex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    location_fopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    location_vopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    location_incidental_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    
    network_capex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_fopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_vopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    
    network_incidental_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    # carbon_emission_location_constraint(instance= instance, network_scale_level= scenario.network_scale_level)
    # carbon_emission_network_constraint(instance= instance, network_scale_level= scenario.network_scale_level)
    

    cost_objective(instance=instance,
                   network_scale_level=scenario.network_scale_level)

    return instance



In [14]:
# milp = formulate_houston_milp(scenario = scenario, demand = 10**3, land_restriction= 6000)
# results = solve(scenario = scenario, instance= milp, solver= 'gurobi', \
#         name=f"FEWN", print_solversteps = True)

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 175290 rows, 131503 columns and 499484 nonzeros
Model fingerprint: 0xf36f6ea0
Variable types: 131495 continuous, 8 integer (8 binary)
Coefficient statistics:
  Matrix range     [4e-03, 1e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+01, 1e+04]
Presolve removed 166525 rows and 122740 columns
Presolve time: 0.63s
Presolved: 8765 rows, 8763 columns, 35050 nonzeros
Variable types: 8761 continuous, 2 integer (2 binary)

Root relaxation: objective 1.005010e+05, 2 iterations, 0.01 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    100500.96700 100500.967  0.00%     -    0s

Explored 1 nodes (2 simplex iterations) in 0.80 seconds (0.90 wo

In [ ]:
# for process in process_set:
#     print(process.land)

In [16]:
def solve_land_restricted(land_restriction):
    milp = formulate_houston_milp(scenario = scenario, demand = 10**3, land_restriction= land_restriction)
    results = solve(scenario = scenario, instance= milp, solver= 'gurobi', \
        name=f"FEWN", print_solversteps = True)
    return results

In [17]:
results_dict = {land: solve_land_restricted(land) for land in [6000 - 100*i for i in range(11)]}

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 175290 rows, 131503 columns and 499484 nonzeros
Model fingerprint: 0xf36f6ea0
Variable types: 131495 continuous, 8 integer (8 binary)
Coefficient statistics:
  Matrix range     [4e-03, 1e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+01, 1e+04]
Presolve removed 166525 rows and 122740 columns
Presolve time: 0.55s
Presolved: 8765 rows, 8763 columns, 35050 nonzeros
Variable types: 8761 continuous, 2 integer (2 binary)

Root relaxation: objective 1.005010e+05, 2 iterations, 0.01 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    100500.96700 100500.967  0.00%     -    0s

Explored 1 nodes (2 simplex iterations) in 0.70 seconds (0.90 wo

In [ ]:
land = [i.output['Land_network'] for i in results_dict.values()]
objective = [i.output['objective'] for i in results_dict.values()]

In [ ]:
milp.Land_location.pprint()

In [ ]:
milp.X_S.pprint()

In [ ]:
results.output['Incidental_network']

In [ ]:
milp.Fopex_network.pprint()

In [ ]:
scenario.demand_factor

In [ ]:
results_reduced_new = solve(scenario = reduced_scenario_new, instance= reduced_milp_new, solver= 'gurobi', \
    name=f"Mobility_reduced_new", print_solversteps = True)

In [ ]:
results_reduced_dtw.__dict__['output']['Fopex_location']

In [ ]:
results_reduced_dtw.__dict__['output']['Vopex_process']

In [ ]:
results_reduced_dtw.__dict__['components'].keys()

In [ ]:
results_reduced_dtw.__dict__['output']['Cap_P']

In [ ]:
plot.schedule(results=results_reduced_dtw, y_axis='Inv',
               component='Charge', location='HO')
plot.schedule(results=results_reduced_dtw, y_axis='Inv',
               component='Charge', location='HO')

# plot.schedule(results=results_EV, y_axis='P',
#                component='WF', location='HO')

# plot.schedule(results=results_EV, y_axis='P',
#                component='EV', location='HO')



In [ ]:
# plot.schedule(results=results_HVG, y_axis='P',
            #    component='AKE', location='HO')
plot.schedule(results=results_reduced_dtw, y_axis='P',
               component='WF', location='HO')
plot.schedule(results=results_reduced_dtw, y_axis='P',
               component='HV', location='HO')
plot.schedule(results=results_reduced_dtw, y_axis='P',
               component='EV', location='HO')

$\textit{Contribution}$

The contribution of different components to a particular variable value can be visualized

In [ ]:

# plot.contribution(results=results_EV, y_axis='P_location', location='HO')
# plot.contribution(results=results_HV, y_axis='P_location', location='HO')
# plot.contribution(results=results_EV, y_axis='S_location', location='HO')
# plot.contribution(results=results_HV, y_axis='S_location', location='HO')
# plot.contribution(results=results_EV, y_axis='B_location', location='HO')
# plot.contribution(results=results_HV, y_axis='B_location', location='HO')
# plot.contribution(results=results_EV, y_axis='Cap_P', location='HO')
# plot.contribution(results=results_HG, y_axis='Cap_P', location='HO')
plot.capacity_utilization(results=results_reduced_dtw, process ='SMR', location='HO')


# plot.contribution(results=results_EV, y_axis='Cap_S', location='HO')
# plot.contribution(results=results_HV, y_axis='Cap_S', location='HO')
# # graph.contribution(results=results_sl, y_axis='P_location', location='HO')


In [ ]:
results_reduced_dtw.output['P_location']